Honors Capstone - 2018 NFCS Analysis
Cleaned Code for Models 1, 2, and 3

In [2]:
# Necessary Imports

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2_contingency

In [5]:
# Survey Settings and Imports

In [6]:
CSV_PATH = "2018_Data.csv"
DK_CODES = [98, 99]
WEIGHT_COL = "WGT1"
ACTIVE_CODES = [2, 3]  

HUMAN_COL = "C22_1"
WEB_COL = "C22_3"
APP_COL = "C22_4"

AGE_COL = "S_Age"
INCOME_COL = "S_Income"
EDU_COL = "S_Education"
ETH_COL = "S_Ethnicity"
GENDER_CANDIDATES = ["S_Gender2", "S_Gender"]

AGE_MAP = {
    1: "18-34",
    2: "35-54",
    3: "55+"
}

In [7]:
# Extra Helper Functions

In [8]:
def load_clean_data(path: str, dk_codes=None) -> pd.DataFrame:
    if dk_codes is None:
        dk_codes = DK_CODES

    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    df.replace({c: np.nan for c in dk_codes}, inplace=True)
    return df


def to_numeric_nan(series: pd.Series, dk_codes=None) -> pd.Series:
    if dk_codes is None:
        dk_codes = DK_CODES
    s = pd.to_numeric(series, errors="coerce")
    return s.replace(dk_codes, np.nan)


def zscore(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    mu = s.mean(skipna=True)
    sd = s.std(skipna=True, ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.nan, index=s.index)
    return (s - mu) / sd

def weighted_mean(x: pd.Series, w: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    mask = x.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float((x[mask] * w[mask]).sum() / w[mask].sum())


def require_columns(df: pd.DataFrame, cols: list[str]) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")


def pick_gender_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No gender column found. Tried: {candidates}")

def classify_advisory_type(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    uses_human = out[HUMAN_COL].isin(ACTIVE_CODES)
    uses_digital = out[WEB_COL].isin(ACTIVE_CODES) | out[APP_COL].isin(ACTIVE_CODES)

    out["advisory_type"] = "unclassified"
    out.loc[uses_human & (~uses_digital), "advisory_type"] = "traditional"
    out.loc[(~uses_human) & uses_digital, "advisory_type"] = "robo"
    out.loc[uses_human & uses_digital, "advisory_type"] = "hybrid"

    return out


def keep_classified(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["advisory_type"].isin(["traditional", "hybrid", "robo"])].copy()

def print_weighted_means(df: pd.DataFrame, columns: list[str], weight_col: str = WEIGHT_COL) -> None:
    if weight_col not in df.columns:
        print("\nNo weight column found; skipping weighted means.")
        return

    w = df[weight_col]
    print("\nWeighted means (overall, WGT1):")
    for col in columns:
        print(f"{col}: {weighted_mean(df[col], w):.4f}")


def print_weighted_means_by_group(
    df: pd.DataFrame,
    group_col: str,
    columns: list[str],
    groups: list[str] | None = None,
    weight_col: str = WEIGHT_COL
) -> None:

    if weight_col not in df.columns:
        print("\nNo weight column found; skipping weighted subgroup means.")
        return

    if groups is None:
        groups = sorted(df[group_col].dropna().unique())
        
        w = df[weight_col]
        print(f"\nWeighted means by {group_col}:")
        for grp in groups:
            mask = df[group_col] == grp
            print(f"\n{grp}:")
            for col in columns:
                print(f"  {col}: {weighted_mean(df.loc[mask, col], w.loc[mask]):.4f}")

In [14]:
# Load data

In [16]:
df = load_clean_data(CSV_PATH)
gender_col = pick_gender_col(df, GENDER_CANDIDATES)
df = classify_advisory_type(df)
df_classified = keep_classified(df)

In [18]:
# MODEL 1: Demographic Predictors from Advisory Type

In [20]:
def subgroup_crosstab(df: pd.DataFrame, group_col: str, group_map: dict | None = None):
    tmp = df.copy()
    tmp["group_label"] = tmp[group_col].map(group_map) if group_map else tmp[group_col]
    tmp = tmp.dropna(subset=["group_label", "advisory_type"])

    counts = pd.crosstab(tmp["group_label"], tmp["advisory_type"])
    row_pct = counts.div(counts.sum(axis=1), axis=0) * 100

    if counts.shape[0] >= 2 and counts.shape[1] >= 2:
        chi2, p, dof, _ = chi2_contingency(counts)
    else:
        chi2, p, dof = np.nan, np.nan, np.nan

    stats = {"chi2": chi2, "p": p, "dof": dof}
    return counts, row_pct, stats


def build_model1_logit_data(df: pd.DataFrame, gender_col: str):
    use = df[[
        "advisory_type", AGE_COL, INCOME_COL, EDU_COL, gender_col, ETH_COL
    ]].dropna().copy()

    use["robo_binary"] = (use["advisory_type"] == "robo").astype(int)

    y = use["robo_binary"]
    X_raw = use[[AGE_COL, INCOME_COL, EDU_COL, gender_col, ETH_COL]].copy()

    for col in X_raw.columns:
        X_raw[col] = X_raw[col].astype("category")

    X = pd.get_dummies(X_raw, drop_first=True)
    X = sm.add_constant(X, has_constant="add").astype(float)

    return y, X, use

def odds_ratio_table(result) -> pd.DataFrame:
    conf = result.conf_int()
    out = pd.DataFrame({
        "term": result.params.index,
        "coef": result.params.values,
        "odds_ratio": np.exp(result.params.values),
        "ci_lower": np.exp(conf[0].values),
        "ci_upper": np.exp(conf[1].values),
        "p_value": result.pvalues.values
    })
    return out.sort_values("p_value")


dist_all = df["advisory_type"].value_counts(dropna=False)
dist_all_pct = (df["advisory_type"].value_counts(normalize=True, dropna=False) * 100).round(2)

age_counts, age_pct, age_stats = subgroup_crosstab(df_classified, AGE_COL, AGE_MAP)
inc_counts, inc_pct, inc_stats = subgroup_crosstab(df_classified, INCOME_COL)
edu_counts, edu_pct, edu_stats = subgroup_crosstab(df_classified, EDU_COL)

y, X, model1_df = build_model1_logit_data(df_classified, gender_col)
model1 = sm.Logit(y, X).fit(disp=False)

or_table = odds_ratio_table(model1)
marginal_effects = (
    model1.get_margeff(at="overall")
    .summary_frame()
    .reset_index()
    .rename(columns={"index": "term"})
)

In [22]:
print("\nMODEL 1 — DEMOGRAPHIC PREDICTORS OF ADVISORY TYPE")
print("\nGender column used:")
print(gender_col)

print("\n1) Advisory Type Distribution (All Cases)")
print(pd.DataFrame({"count": dist_all, "percent": dist_all_pct}))

print("\n2) Age vs Advisory Type (Row %)")
print(age_pct.round(2))
print(f"Chi-square = {age_stats['chi2']:.4f}, p = {age_stats['p']:.4f}, dof = {age_stats['dof']}")

print("\n3) Income vs Advisory Type (Row %)")
print(inc_pct.round(2))
print(f"Chi-square = {inc_stats['chi2']:.4f}, p = {inc_stats['p']:.4f}, dof = {inc_stats['dof']}")

print("\n4) Education vs Advisory Type (Row %)")
print(edu_pct.round(2))
print(f"Chi-square = {edu_stats['chi2']:.4f}, p = {edu_stats['p']:.4f}, dof = {edu_stats['dof']}")

print("\n5) Logistic Regression: Predicting Robo Use")
print(f"N used in regression: {len(model1_df)}")
print(f"Pseudo R-squared: {model1.prsquared:.4f}")

print("\nOdds Ratios")
print(or_table.round(4))

print("\nMarginal Effects")
print(marginal_effects.round(4))


MODEL 1 — DEMOGRAPHIC PREDICTORS OF ADVISORY TYPE

Gender column used:
S_Gender

1) Advisory Type Distribution (All Cases)
               count  percent
advisory_type                
robo             585    29.21
traditional      534    26.66
hybrid           495    24.71
unclassified     389    19.42

2) Age vs Advisory Type (Row %)
advisory_type  hybrid   robo  traditional
group_label                              
18-34           59.05  30.00        10.95
35-54           40.44  40.22        19.33
55+             19.81  35.74        44.44
Chi-square = 204.1455, p = 0.0000, dof = 4

3) Income vs Advisory Type (Row %)
advisory_type  hybrid   robo  traditional
group_label                              
1               24.73  43.09        32.18
2               33.88  33.73        32.38
3               30.82  34.68        34.50
Chi-square = 13.5099, p = 0.0090, dof = 4

4) Education vs Advisory Type (Row %)
advisory_type  hybrid   robo  traditional
group_label                              

In [24]:
# MODEL 2: Planning Index

In [26]:
def require_columns(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

def to_numeric_nan(series, dk_codes=[98, 99]):
    s = pd.to_numeric(series, errors="coerce")
    return s.replace(dk_codes, np.nan)

def zscore(series):
    s = pd.to_numeric(series, errors="coerce")
    mu = s.mean(skipna=True)
    sd = s.std(skipna=True, ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.nan, index=s.index)
    return (s - mu) / sd

def weighted_mean(x, w):
    x = pd.to_numeric(x, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    mask = x.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float((x[mask] * w[mask]).sum() / w[mask].sum())


COMFORT_COL = "G1"
KNOWLEDGE_COL = "G2"
INFOACCESS_COL = "G20"
TRADE_FREQ_COL = "B3"

require_columns(df, [COMFORT_COL, KNOWLEDGE_COL, INFOACCESS_COL, TRADE_FREQ_COL])

model2_df = df_classified.copy()

model2_df["b3_trade_rev"] = 5 - to_numeric_nan(model2_df[TRADE_FREQ_COL])

model2_df["z_comfort"] = zscore(to_numeric_nan(model2_df[COMFORT_COL]))
model2_df["z_knowledge"] = zscore(to_numeric_nan(model2_df[KNOWLEDGE_COL]))
model2_df["z_infoaccess"] = zscore(to_numeric_nan(model2_df[INFOACCESS_COL]))
model2_df["z_trade_freq"] = zscore(model2_df["b3_trade_rev"])

planning_components = [
    "z_comfort",
    "z_knowledge",
    "z_infoaccess",
    "z_trade_freq"
]

model2_df["planning_index"] = np.where(
    model2_df[planning_components].notna().all(axis=1),
    model2_df[planning_components].mean(axis=1),
    np.nan
)

n = int(model2_df["planning_index"].notna().sum())
mean_unw = float(model2_df["planning_index"].mean(skipna=True))

In [28]:
print("\nMODEL 2 — PREPARDNESS/PLANNING INDEX (2018)")
print("-----------------------------------")
print(f"Non-missing N: {n}")
print(f"Unweighted overall mean: {mean_unw:.4f}")

if WEIGHT_COL in model2_df.columns:
    print(f"Weighted overall mean (WGT1): {weighted_mean(model2_df['planning_index'], model2_df[WEIGHT_COL]):.4f}")
else:
    print("No weight column found; skipping weighted mean.")

print("\nUnweighted mean by advisory type:")
print(
    model2_df.groupby("advisory_type")["planning_index"]
    .mean()
    .reindex(["traditional", "hybrid", "robo"])
    .round(4)
)

print("\nWeighted mean by advisory type:")
for grp in ["traditional", "hybrid", "robo"]:
    sub = model2_df[model2_df["advisory_type"] == grp]
    print(f"{grp}: {weighted_mean(sub['planning_index'], sub[WEIGHT_COL]):.4f}")

print("\nComponents used:")
print(planning_components)


MODEL 2 — PREPARDNESS/PLANNING INDEX (2018)
-----------------------------------
Non-missing N: 1524
Unweighted overall mean: 0.0261
Weighted overall mean (WGT1): 0.0228

Unweighted mean by advisory type:
advisory_type
traditional   -0.0315
hybrid         0.1150
robo          -0.0010
Name: planning_index, dtype: float64

Weighted mean by advisory type:
traditional: -0.0511
hybrid: 0.1297
robo: -0.0361

Components used:
['z_comfort', 'z_knowledge', 'z_infoaccess', 'z_trade_freq']


In [30]:
# MODEL 3: Bounded Rationality Index

In [32]:
model3_df = df_classified.copy()

LITERACY_KEY = {
    "G4": 1,
    "G5": 2,
    "G6": 2,
    "G7": 1,
    "G8": 1,
    "G21": 2,
    "G22": 2,
}

lit_items = [c for c in LITERACY_KEY if c in model3_df.columns]
if len(lit_items) == 0:
    raise ValueError("None of the expected 2018 literacy columns were found.")

lit_correct = []
for col in lit_items:
    lit_correct.append((model3_df[col] == LITERACY_KEY[col]).astype(float))

lit_df = pd.concat(lit_correct, axis=1)
model3_df["literacy_score"] = lit_df.sum(axis=1, min_count=1)
model3_df["literacy_pct"] = lit_df.mean(axis=1)

HEURISTIC_COLS = ["F1_9", "F2_6", "F2_22"]
heuristic_items = [c for c in HEURISTIC_COLS if c in model3_df.columns]

if len(heuristic_items) == 0:
    model3_df["heuristic_reliance"] = np.nan
else:
    recoded = []
    for c in heuristic_items:
        s = model3_df[c]
        recoded.append(pd.Series(np.where(s == 1, 1, np.where(s == 2, 0, np.nan)), index=model3_df.index))
    model3_df["heuristic_reliance"] = pd.concat(recoded, axis=1).mean(axis=1)

SELF_RATED_KNOW_COL = "G2"
if SELF_RATED_KNOW_COL not in model3_df.columns:
    raise ValueError("Missing G2 (self-rated investment knowledge).")

model3_df["self_rated_knowledge"] = model3_df[SELF_RATED_KNOW_COL]

model3_df["overconfidence_gap"] = (
    zscore(model3_df["self_rated_knowledge"]) -
    zscore(model3_df["literacy_score"])
)

model3_df["BRI"] = (
    (-1 * zscore(model3_df["literacy_score"])) +
    zscore(model3_df["heuristic_reliance"]) +
    zscore(model3_df["overconfidence_gap"])
) / 3

In [34]:
print("\nMODEL 3 — BOUNDED RATIONALITY INDEX")
print("\nAdvisory type counts:")
print(model3_df["advisory_type"].value_counts())

print("\nNon-missing counts:")
print(
    model3_df[[
        "literacy_score",
        "heuristic_reliance",
        "overconfidence_gap",
        "BRI"
    ]].notna().sum()
)

print("\nUnweighted means by advisory type:")
print(
    model3_df.groupby("advisory_type")[[
        "literacy_score",
        "heuristic_reliance",
        "overconfidence_gap",
        "BRI"
    ]].mean(numeric_only=True)
)

print_weighted_means(
    model3_df,
    ["literacy_score", "heuristic_reliance", "overconfidence_gap", "BRI"]
)

print_weighted_means_by_group(
    model3_df,
    group_col="advisory_type",
    groups=["traditional", "hybrid", "robo"],
    columns=["literacy_score", "heuristic_reliance", "overconfidence_gap", "BRI"]
)


MODEL 3 — BOUNDED RATIONALITY INDEX

Advisory type counts:
advisory_type
robo           585
traditional    534
hybrid         495
Name: count, dtype: int64

Non-missing counts:
literacy_score        1614
heuristic_reliance    1609
overconfidence_gap    1608
BRI                   1603
dtype: int64

Unweighted means by advisory type:
               literacy_score  heuristic_reliance  overconfidence_gap  \
advisory_type                                                           
hybrid               4.042424            0.484079            0.462560   
robo                 4.692308            0.390223           -0.263134   
traditional          4.011236            0.205056           -0.141832   

                    BRI  
advisory_type            
hybrid         0.302795  
robo          -0.121065  
traditional   -0.149284  

Weighted means (overall, WGT1):
literacy_score: 4.0475
heuristic_reliance: 0.3946
overconfidence_gap: 0.1365
BRI: 0.1189
